In [ ]:
#!pip install astropy

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits
from sklearn.cluster import KMeans

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/good_parents_fit.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    data = hdu_list[1].data
    header = hdu_list[1].header
print(len(data), header)

In [ ]:
features = data["features"]
print(features.shape)

#create the time invariant feature vector for clustering (log(a0^2), log(a1^2 + b1^2), ...)
squared_feats = np.zeros((1438, 33))
for e,f in enumerate(features):
    j = 0
    for i in range(len(f) - 1):
        if i == 0:
            squared_feats[e][j] = f[i]**2
            j = j + 1
        if i%2 == 1:
            squared_feats[e][j] = f[i]**2 + f[i+1]**2
            j = j + 1


## plotting

In [ ]:
#from hogg
foo, k = squared_feats.shape
frequencies = np.outer(data["refined_frequency"], (1. + np.arange(k)))
informations = np.nansum(squared_feats * frequencies * (frequencies < 24.), axis=1)
print(informations)
refine_freqs = data["refined_frequency"]

In [ ]:
#from hogg
sizes = 0.5 * np.log10(informations)
sizes += 4.
sizes = np.clip(sizes, 0.01, None)
print(sizes)
for i, j in [(0, 1),
             (0, 2),
             (0, 3),
             (1, 2),(2,3),(1,3)]:
    plt.axhline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.axvline(1.0, color="k", lw=1.0, alpha=0.5)
    plt.scatter(squared_feats[:, i], squared_feats[:, j], c=np.log10(refine_freqs), s=sizes)
    plt.loglog()
    plt.xlabel(f"scalar {i}")
    plt.ylabel(f"scalar {j}")
    plt.colorbar(label="log_10 frequency")
    plt.savefig(f"scatter_{i}_{j}.png")
    plt.show()

## running K-means

### k = 3

In [ ]:
#run with k = 3,10,30 with 3 random restarts, 9 plots total


#some stars with NaN values
good_mask = ~np.any(np.isnan(squared_feats), axis=1)
print(f"keeping {good_mask.sum()} of {len(squared_feats)} stars")

squared_feats_clean = squared_feats[good_mask]
log_squared_feats = np.log(squared_feats_clean)
log_squared_feats = log_squared_feats[:, 1:] #get rid of constant?

randoms = np.array([42, 33,50])
for r in randoms:
    kmeans = KMeans(n_clusters=3, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)  
    y_kmeans = kmeans.predict(log_squared_feats)
    
    centroids = kmeans.cluster_centers_  
    labels = kmeans.labels_              
    
    plt.scatter(log_squared_feats[:, 0], log_squared_feats[:, 1], 
                c=y_kmeans, 
                s=2,          
                cmap='viridis')
    plt.colorbar()
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=25, alpha=0.8, marker='o')
    plt.title(f"k = 3, random restart {r}")
    plt.xlabel("log scalar 1")
    plt.ylabel("log scalar 2")
    plt.show()

### k = 10

In [ ]:
for r in randoms:
    kmeans = KMeans(n_clusters=10, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)  
    y_kmeans = kmeans.predict(log_squared_feats)
    
    centroids = kmeans.cluster_centers_  
    labels = kmeans.labels_              
    
    plt.scatter(log_squared_feats[:, 0], log_squared_feats[:, 1], 
                c=y_kmeans, 
                s=2,          
                cmap='viridis')
    plt.colorbar()
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=15, alpha=0.8, marker='o')
    plt.title(f"k = 10, random restart {r}")
    plt.xlabel("log scalar 1")
    plt.ylabel("log scalar 2")
    plt.show()

### k = 30

In [ ]:
for r in randoms:
    kmeans = KMeans(n_clusters=30, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)  
    y_kmeans = kmeans.predict(log_squared_feats)
    
    centroids = kmeans.cluster_centers_  
    labels = kmeans.labels_              
    
    plt.scatter(log_squared_feats[:, 0], log_squared_feats[:, 1], 
                c=y_kmeans, 
                s=2,          
                cmap='viridis')
    plt.colorbar()
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=15, alpha=0.8, marker='o')
    plt.title(f"k = 30, random restart {r}")
    plt.xlabel("log scalar 1")
    plt.ylabel("log scalar 2")
    plt.show()